# Notebook 01: ML Threat Modeling

## Why This Matters
Before you can defend a machine learning system, you need a precise vocabulary for what can go wrong and where. ML security is a distinct discipline from classical software security — the attack surfaces are different, the failure modes are different, and the mitigations are different. A SQL injection scanner won't find a backdoored training set. A firewall won't stop membership inference against your model weights.

This notebook builds the threat modeling foundation for the entire series. By the end, you'll have a structured mental model of the ML attack surface — one that maps cleanly to the 20 notebooks that follow.

## Structure
1. What is ML threat modeling?
2. The four attack surfaces: training, inference, supply chain, deployment
3. Attack taxonomy: goals, capabilities, knowledge
4. Concrete attack families — one per surface
5. STRIDE applied to ML systems
6. A worked example: threat modeling a real RAG pipeline
7. Building a threat model template you can reuse

## What You'll Learn
- Why ML security is categorically different from classical application security
- The four attack surfaces every ML system exposes, and which attacks target each
- How to characterize an attacker's goal, capability, and knowledge — the three axes that define threat severity
- How to run a structured threat modeling session on any ML pipeline
- Which notebooks in this series address each threat category

## Background

### Why ML security is a different discipline

Classical application security has a 30-year body of practice. We know the OWASP Top 10. We know how to find buffer overflows, SQL injections, and authentication bypasses. Tools exist for static analysis, fuzz testing, and penetration testing at every layer of the stack.

ML systems break most of these assumptions.

In a classical system, the logic is in the code — you audit the code. In an ML system, a significant fraction of the logic is in the weights, learned from data. You can't audit weights with a code scanner. You can't grep for a backdoor in a neural network the way you can grep for `eval(user_input)`.

In a classical system, the attack boundary is inputs and outputs. In an ML system, the attack boundary extends backwards through the entire training pipeline — the dataset, the training code, the pretrained model you fine-tuned from, the Python packages you used to train. An attacker who can touch any of these can affect the model's behavior in production without ever touching a single line of your application code.

In a classical system, unintended behavior is a bug — it happens because the developer made an error. In an ML system, unintended behavior can be *deliberately engineered* by someone who manipulated the training data or the model weights. The system does exactly what it was trained to do; the problem is that it was trained adversarially.

This is why ML security emerged as a distinct field, rather than being absorbed into traditional AppSec.

### A brief history of the field

The field has two origins that converged:

**Adversarial examples (2013–2014).** Szegedy et al. (2013) showed that neural networks could be fooled by imperceptible perturbations to inputs — images that look identical to humans but cause catastrophic misclassification. Goodfellow et al. (2014) introduced FGSM (Fast Gradient Sign Method), the first practical algorithm for generating adversarial examples. This line of work dominated ML security research for the next five years and is still active.

**Privacy attacks (2016–2019).** Shokri et al. (2017) introduced membership inference attacks — given black-box access to a model, can you determine whether a specific record was in its training set? Fredrikson et al. (2015) showed model inversion — reconstructing training inputs from model outputs. These attacks demonstrated that models leak information about their training data in ways that weren't previously understood.

**Training-time attacks (2017–present).** Chen et al. (2017) introduced neural backdoors (BadNets) — poisoning a training dataset with trigger patterns that cause specific misclassification at test time. Biggio et al. had earlier work on poisoning in classical ML. This line established that the *training process* itself is an attack surface.

**Supply chain and deployment attacks (2020–present).** As models became widely shared via HuggingFace and other hubs, supply chain attacks became real. Malicious pickle files, backdoored pretrained models, and ONNX graph injection became practical concerns. The 2022 disclosure of a backdoored model on HuggingFace Hub made this concrete.

Today, ML security has a dedicated track at every major security conference (IEEE S&P, Usenix Security, CCS, NDSS) and a growing practitioner community through venues like MLSEC workshop, TCB (Trustworthy and Reliable ML), and SaTML.

### Why most teams don't do ML threat modeling

The gap between knowledge and practice is wide. Most ML teams:
- Are aware of adversarial examples (everyone has seen the panda-gibbon image)
- Have not thought about their training data pipeline as an attack surface
- Have not considered what their model reveals about its training set
- Have not audited pretrained models before loading them
- Have no incident response plan for "model is behaving unexpectedly"

The disconnect exists partly because ML practitioners come from a research and statistics background, not a security background, and the threat modeling vocabulary never made it into ML education. This series aims to close that gap.

### The key insight: attacks happen at different stages

The most important structuring principle in ML threat modeling is **when in the ML lifecycle does the attack occur?** This determines what the attacker needs access to, what the attacker can achieve, and what defenses are possible.

An attacker who can manipulate your training data operates under completely different constraints and achieves completely different goals than an attacker who queries your deployed model API. Conflating them leads to threat models that are simultaneously over-scoped ("we need to defend against everything") and under-scoped ("we're protected because we validated our API inputs").

In [ ]:
%pip install -q numpy pandas matplotlib networkx

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from dataclasses import dataclass, field
from typing import List
from enum import Enum

plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
print("Imports OK")

## 1. The Four Attack Surfaces

Every ML system exposes four distinct attack surfaces. Understanding which surface a threat targets tells you immediately:
- What access the attacker needs
- When in the pipeline the attack occurs
- What the attacker can achieve
- What defenses apply

```
┌─────────────────────────────────────────────────────────────────┐
│  ML SYSTEM ATTACK SURFACES                                      │
│                                                                 │
│  [Data Collection] → [Training] → [Model] → [Deployment/API]   │
│        │                 │            │              │          │
│   Supply Chain      Training-Time   Supply       Inference     │
│    Attacks          Attacks         Chain         Attacks       │
│  (data poison,    (backdoor,       Attacks      (adversarial   │
│   label flip)     poisoning)    (model theft,    examples,     │
│                                  pickle bomb)    membership    │
│                                                  inference)    │
└─────────────────────────────────────────────────────────────────┘
```

| Surface | Attacker access | Attack timing | Primary threats |
|---------|----------------|---------------|-----------------|
| **Training data** | Write access to dataset or labels | Before/during training | Poisoning, backdoors, label flips |
| **Training process** | Write access to training code or environment | During training | Gradient manipulation, supply chain compromise |
| **Model / supply chain** | Read/write access to weights or artifacts | After training, before deployment | Backdoor implant, pickle exploits, ONNX injection |
| **Inference / deployment** | Query access to deployed model | During serving | Adversarial examples, membership inference, model stealing |

In [ ]:
# Encode the attack surface taxonomy as structured data
# We'll use this throughout the notebook and build on it for threat modeling

class AttackSurface(Enum):
    TRAINING_DATA = "Training Data"
    TRAINING_PROCESS = "Training Process"
    MODEL_SUPPLY_CHAIN = "Model / Supply Chain"
    INFERENCE = "Inference / Deployment"


@dataclass
class AttackFamily:
    name: str
    surface: AttackSurface
    attacker_goal: str           # What the attacker achieves
    attacker_capability: str     # What the attacker needs access to
    attacker_knowledge: str      # What the attacker needs to know
    impact: str                  # CIA-style impact
    notebook: int                # Which notebook in this series covers it
    mitigations: List[str] = field(default_factory=list)


ATTACK_TAXONOMY = [
    AttackFamily(
        name="Data Poisoning",
        surface=AttackSurface.TRAINING_DATA,
        attacker_goal="Degrade model accuracy or cause targeted misclassification",
        attacker_capability="Write access to training dataset or label pipeline",
        attacker_knowledge="Dataset distribution; optionally model architecture",
        impact="Integrity — model produces wrong outputs",
        notebook=10,
        mitigations=["Dataset provenance tracking", "Anomaly detection on labels", "Influence function auditing"],
    ),
    AttackFamily(
        name="Backdoor / Trojan Attack",
        surface=AttackSurface.TRAINING_DATA,
        attacker_goal="Implant a secret trigger that causes targeted behavior",
        attacker_capability="Write access to training data (can be small fraction)",
        attacker_knowledge="Trigger pattern; target class",
        impact="Integrity — model behaves normally except on trigger inputs",
        notebook=8,
        mitigations=["Neural Cleanse", "STRIP detection", "Activation clustering"],
    ),
    AttackFamily(
        name="Membership Inference",
        surface=AttackSurface.INFERENCE,
        attacker_goal="Determine if a specific record was in the training set",
        attacker_capability="Black-box query access to model API",
        attacker_knowledge="The candidate record; optionally model architecture",
        impact="Privacy — training data exposure, regulatory risk",
        notebook=5,
        mitigations=["Differential privacy training", "Confidence score masking", "Prediction smoothing"],
    ),
    AttackFamily(
        name="Model Inversion",
        surface=AttackSurface.INFERENCE,
        attacker_goal="Reconstruct representative training samples from model outputs",
        attacker_capability="White-box or black-box model access",
        attacker_knowledge="Model architecture (white-box) or just API access (black-box)",
        impact="Privacy — reconstruction of sensitive training data",
        notebook=6,
        mitigations=["Differential privacy", "Output perturbation", "API rate limiting"],
    ),
    AttackFamily(
        name="Adversarial Examples",
        surface=AttackSurface.INFERENCE,
        attacker_goal="Cause misclassification via imperceptible input perturbation",
        attacker_capability="Query access to model; ability to manipulate input",
        attacker_knowledge="White-box (gradient access) or black-box (query-only)",
        impact="Integrity — targeted or untargeted misclassification",
        notebook=2,
        mitigations=["Adversarial training", "Input preprocessing", "Certified defenses"],
    ),
    AttackFamily(
        name="Model Stealing",
        surface=AttackSurface.INFERENCE,
        attacker_goal="Extract a functionally equivalent model via API queries",
        attacker_capability="Query access to model API",
        attacker_knowledge="Input domain; optionally model architecture",
        impact="Confidentiality — IP theft, enables downstream attacks",
        notebook=12,
        mitigations=["Rate limiting", "Output perturbation", "Watermarking"],
    ),
    AttackFamily(
        name="Supply Chain Compromise",
        surface=AttackSurface.MODEL_SUPPLY_CHAIN,
        attacker_goal="Inject malicious behavior or malware via shared model artifacts",
        attacker_capability="Write access to model registry or package repository",
        attacker_knowledge="Target environment; artifact format (pickle, ONNX, safetensors)",
        impact="Integrity / RCE — backdoor behavior or arbitrary code execution on load",
        notebook=11,
        mitigations=["Artifact scanning", "Hash verification", "Prefer safetensors over pickle"],
    ),
    AttackFamily(
        name="Attribute Inference",
        surface=AttackSurface.INFERENCE,
        attacker_goal="Infer sensitive attributes not included in model inputs",
        attacker_capability="Black-box API access + partial knowledge of a target record",
        attacker_knowledge="Model's feature space; correlation structure of training data",
        impact="Privacy — exposure of sensitive attributes (race, health, income)",
        notebook=7,
        mitigations=["Fairness constraints", "Feature removal", "Output regularization"],
    ),
]

print(f"Loaded {len(ATTACK_TAXONOMY)} attack families across {len(AttackSurface)} surfaces")
print()
for surface in AttackSurface:
    attacks = [a for a in ATTACK_TAXONOMY if a.surface == surface]
    print(f"  {surface.value}: {len(attacks)} attack families")

## 2. Attacker Characterization: Goals, Capabilities, Knowledge

Threat severity depends on three axes:

**Goal** — what does the attacker want?
- *Integrity:* cause wrong outputs (misclassification, injected behavior)
- *Confidentiality:* extract information (training data, model weights, membership)
- *Availability:* degrade model performance or cause denial of service

**Capability** — what can the attacker do?
- *Write access to training data:* the most dangerous capability; enables backdoors and poisoning
- *Query access to deployed model:* enables inference attacks and model stealing
- *Read access to model weights:* enables white-box attacks, more powerful than query-only
- *Write access to model artifacts:* enables supply chain attacks

**Knowledge** — what does the attacker know?
- *White-box:* full model architecture and weights — strongest assumption, best-case for attacker
- *Gray-box:* knows architecture but not weights (common — architecture often published)
- *Black-box:* query access only — most realistic for deployed APIs

The intersection of goal × capability × knowledge defines the realistic threat. Most real-world attackers are black-box against deployed models (they query your API, they don't have your weights). But a supply chain attacker may have white-box access to the pretrained model they're compromising.

In [ ]:
# Visualize the attack surface landscape
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: attacks by surface
surface_counts = {}
for attack in ATTACK_TAXONOMY:
    s = attack.surface.value
    surface_counts[s] = surface_counts.get(s, 0) + 1

colors = ['#e74c3c', '#e67e22', '#3498db', '#2ecc71']
bars = axes[0].barh(list(surface_counts.keys()), list(surface_counts.values()), color=colors)
axes[0].set_xlabel("Number of Attack Families")
axes[0].set_title("Attack Families by Surface\n(this series)")
axes[0].set_xlim(0, max(surface_counts.values()) + 1)
for bar, val in zip(bars, surface_counts.values()):
    axes[0].text(bar.get_width() + 0.05, bar.get_y() + bar.get_height()/2,
                 str(val), va='center')

# Right: attacker knowledge vs capability heatmap
# Which attacks require which level of access?
capabilities = [
    "Query API\n(black-box)",
    "Model weights\n(white-box)",
    "Write training\ndata",
    "Write model\nartifacts",
]
attack_names = [a.name for a in ATTACK_TAXONOMY]

# Manual mapping: which capability does each attack require?
capability_map = {
    "Data Poisoning":          [0, 0, 1, 0],
    "Backdoor / Trojan Attack": [0, 0, 1, 0],
    "Membership Inference":    [1, 0, 0, 0],
    "Model Inversion":         [1, 1, 0, 0],
    "Adversarial Examples":    [1, 1, 0, 0],
    "Model Stealing":          [1, 0, 0, 0],
    "Supply Chain Compromise": [0, 0, 0, 1],
    "Attribute Inference":     [1, 0, 0, 0],
}

matrix = np.array([capability_map[a] for a in attack_names])
im = axes[1].imshow(matrix, cmap='RdYlGn', aspect='auto', vmin=0, vmax=1)
axes[1].set_xticks(range(len(capabilities)))
axes[1].set_xticklabels(capabilities, fontsize=8)
axes[1].set_yticks(range(len(attack_names)))
axes[1].set_yticklabels(attack_names, fontsize=8)
axes[1].set_title("Attack Capability Requirements\n(green = required)")

plt.tight_layout()
plt.savefig("attack_surface_overview.png", bbox_inches='tight')
plt.show()

## 3. STRIDE Applied to ML Systems

STRIDE is a Microsoft threat modeling framework from the late 1990s — six threat categories that cover most classical software security concerns. It maps cleanly onto ML systems with one addition: **Model Integrity** as a distinct concern separate from Data Integrity.

| STRIDE category | Classical example | ML equivalent |
|-----------------|-------------------|---------------|
| **S**poofing | Forged authentication token | Adversarial example that evades identity verification |
| **T**ampering | Modifying database records | Poisoning training data; backdooring model weights |
| **R**epudiation | Deleting audit logs | Model decisions that can't be attributed or explained |
| **I**nformation Disclosure | Reading encrypted traffic | Membership inference; model inversion; attribute inference |
| **D**enial of Service | SYN flood | Adversarial inputs that cause catastrophic inference cost (sponge examples) |
| **E**levation of Privilege | Privilege escalation exploit | Model used beyond its intended scope; fine-tuning to remove safety guardrails |

STRIDE is useful for stakeholder communication — it maps ML-specific threats to familiar security vocabulary, which makes threat modeling sessions more productive with non-ML security teams.

In [ ]:
# Encode STRIDE mapping for ML
stride_ml_mapping = {
    "Spoofing":                ["Adversarial examples in biometric systems",
                                "Voice cloning to bypass speaker verification",
                                "Deepfake for identity fraud"],
    "Tampering":               ["Training data poisoning",
                                "Backdoor implantation via supply chain",
                                "Model weight tampering post-deployment"],
    "Repudiation":             ["Black-box model decisions without audit trails",
                                "Inability to explain why a prediction was made",
                                "No versioning of model + data used for decision"],
    "Information Disclosure":  ["Membership inference against training set",
                                "Model inversion to reconstruct training data",
                                "Attribute inference from model outputs"],
    "Denial of Service":       ["Sponge examples: inputs maximizing inference compute",
                                "OOM via adversarial long-context inputs (LLMs)",
                                "Degrading model quality to make it unusable"],
    "Elevation of Privilege":  ["Fine-tuning away safety guardrails",
                                "Using model beyond intended scope",
                                "Jailbreaking LLMs to bypass access controls"],
}

print("STRIDE → ML Threat Mapping")
print("=" * 60)
for category, threats in stride_ml_mapping.items():
    print(f"\n[{category}]")
    for t in threats:
        print(f"  • {t}")

## 4. Worked Example: Threat Modeling a RAG Pipeline

Let's apply this framework to a realistic system: a Retrieval-Augmented Generation (RAG) pipeline. This is one of the most commonly deployed ML architectures today — an LLM that retrieves context from a vector database before generating a response.

```
User Query
    │
    ▼
Embedding Model ──→ Vector Database ──→ Retrieved Chunks
                         │                      │
                    [Attack surface:        [Attack surface:
                     embedding model,        RAG poisoning]
                     index integrity]
                                               │
                                               ▼
                                          LLM (Generator)
                                         [Attack surface:
                                          prompt injection,
                                          model theft,
                                          membership inference]
                                               │
                                               ▼
                                          Response
```

This architecture has attack surfaces that don't exist in a standalone LLM deployment. The vector database is a new component with its own integrity requirements. The embedding model is a dependency that could be supply-chain compromised. Retrieved chunks are untrusted external content that flow directly into the LLM's context window.

In [ ]:
@dataclass
class Threat:
    id: str
    component: str
    stride_category: str
    description: str
    attacker_capability: str
    impact: str
    likelihood: str          # Low / Medium / High
    severity: str            # Low / Medium / High / Critical
    mitigation: str


rag_threats = [
    Threat(
        id="T-01",
        component="Vector Database",
        stride_category="Tampering",
        description="Attacker injects malicious documents into the knowledge base that contain prompt injection payloads",
        attacker_capability="Write access to document ingestion pipeline",
        impact="LLM follows attacker instructions embedded in retrieved context",
        likelihood="Medium",
        severity="High",
        mitigation="Input sanitization on ingested documents; prompt injection classifiers on retrieved chunks",
    ),
    Threat(
        id="T-02",
        component="Embedding Model",
        stride_category="Tampering",
        description="Supply chain compromise of the embedding model introduces a backdoor that causes specific queries to retrieve attacker-chosen content",
        attacker_capability="Control over model registry or package upstream",
        impact="Attacker can steer retrieval for targeted queries without touching the database",
        likelihood="Low",
        severity="Critical",
        mitigation="Hash verification of model artifacts; prefer safetensors; scan models before loading",
    ),
    Threat(
        id="T-03",
        component="LLM Generator",
        stride_category="Information Disclosure",
        description="Attacker uses membership inference to determine if specific documents were in the LLM pretraining set",
        attacker_capability="Black-box query access to API",
        impact="Exposure of sensitive pretraining data; regulatory risk under GDPR",
        likelihood="Medium",
        severity="Medium",
        mitigation="Differential privacy at training time; confidence score masking; right-to-erasure protocols",
    ),
    Threat(
        id="T-04",
        component="LLM Generator",
        stride_category="Elevation of Privilege",
        description="Attacker crafts queries that cause the LLM to reveal system prompt or bypass access controls",
        attacker_capability="API query access",
        impact="System prompt leakage; unauthorized capability access",
        likelihood="High",
        severity="High",
        mitigation="Prompt hardening; canary tokens in system prompt; output classifiers",
    ),
    Threat(
        id="T-05",
        component="API / Serving Layer",
        stride_category="Denial of Service",
        description="Attacker submits maximally-long context inputs to exhaust GPU memory or KV cache",
        attacker_capability="API query access",
        impact="Service degradation or outage; cost amplification",
        likelihood="Medium",
        severity="Medium",
        mitigation="Context length limits; rate limiting; cost-per-token monitoring",
    ),
]

# Display as a threat register
df = pd.DataFrame([
    {
        "ID": t.id,
        "Component": t.component,
        "STRIDE": t.stride_category,
        "Likelihood": t.likelihood,
        "Severity": t.severity,
        "Description": t.description[:60] + "...",
    }
    for t in rag_threats
])

print("RAG Pipeline Threat Register")
print(df.to_string(index=False))

In [ ]:
# Risk matrix: likelihood vs severity
likelihood_map = {"Low": 1, "Medium": 2, "High": 3}
severity_map = {"Low": 1, "Medium": 2, "High": 3, "Critical": 4}

fig, ax = plt.subplots(figsize=(8, 6))

# Draw risk zones
# Green: low risk (bottom-left), Red: high risk (top-right)
risk_colors = [
    (1, 1, '#2ecc71'), (1, 2, '#f39c12'), (1, 3, '#f39c12'), (1, 4, '#e74c3c'),
    (2, 1, '#f39c12'), (2, 2, '#f39c12'), (2, 3, '#e74c3c'), (2, 4, '#e74c3c'),
    (3, 1, '#f39c12'), (3, 2, '#e74c3c'), (3, 3, '#e74c3c'), (3, 4, '#c0392b'),
]
for lk, sv, color in risk_colors:
    ax.add_patch(plt.Rectangle((lk-0.5, sv-0.5), 1, 1, color=color, alpha=0.2, zorder=0))

# Plot threats
jitter = np.random.RandomState(42)
for t in rag_threats:
    x = likelihood_map[t.likelihood] + jitter.uniform(-0.1, 0.1)
    y = severity_map[t.severity] + jitter.uniform(-0.1, 0.1)
    ax.scatter(x, y, s=200, zorder=3, color='#2c3e50')
    ax.annotate(t.id, (x, y), textcoords="offset points", xytext=(8, 4), fontsize=9, fontweight='bold')

ax.set_xlim(0.5, 3.5)
ax.set_ylim(0.5, 4.5)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['Low', 'Medium', 'High'])
ax.set_yticks([1, 2, 3, 4])
ax.set_yticklabels(['Low', 'Medium', 'High', 'Critical'])
ax.set_xlabel("Likelihood")
ax.set_ylabel("Severity")
ax.set_title("RAG Pipeline Risk Matrix")
ax.grid(True, alpha=0.3)

legend_elements = [
    mpatches.Patch(color='#2ecc71', alpha=0.4, label='Low risk'),
    mpatches.Patch(color='#f39c12', alpha=0.4, label='Medium risk'),
    mpatches.Patch(color='#e74c3c', alpha=0.4, label='High risk'),
    mpatches.Patch(color='#c0392b', alpha=0.4, label='Critical risk'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=8)

plt.tight_layout()
plt.savefig("risk_matrix.png", bbox_inches='tight')
plt.show()

print("\nHighest priority threats (severity × likelihood):")
for t in sorted(rag_threats, key=lambda t: -likelihood_map[t.likelihood] * severity_map[t.severity]):
    score = likelihood_map[t.likelihood] * severity_map[t.severity]
    print(f"  {t.id} ({t.severity}/{t.likelihood}) score={score}: {t.description[:55]}...")

## 5. A Reusable Threat Model Template

The process above is repeatable. For any ML system you want to threat model:

1. **Draw the system boundary** — what components are in scope? Data pipeline, training infra, model artifacts, serving layer, downstream systems?

2. **Identify trust boundaries** — where does data cross from one trust domain to another? User inputs, external APIs, shared model registries, third-party datasets.

3. **Apply STRIDE to each component** — ask: can this component be Spoofed? Tampered with? Repudiated? Does it disclose information? Could it be a DoS target? Does it grant excessive privilege?

4. **Characterize attackers** — for each threat, what goal/capability/knowledge does an attacker need? Is this a realistic attacker for your threat model?

5. **Score and prioritize** — likelihood × severity gives a rough risk score. Address high-severity/high-likelihood threats first.

6. **Identify mitigations** — mitigations exist at the component level (scan model artifacts), pipeline level (audit training data), and policy level (access controls on model registry).

One practical tip: run the threat model in a session with both the ML team and a security engineer. ML teams know the system deeply but often haven't thought adversarially. Security engineers know attack patterns but often don't know what's technically possible against ML models.

In [ ]:
def run_threat_model_checklist(system_name: str, components: list[str]) -> None:
    """
    Interactive threat model checklist.
    Prints structured questions to guide a threat modeling session.
    """
    print(f"Threat Model: {system_name}")
    print("=" * 60)
    print()

    # Training data surface
    print("[ TRAINING DATA SURFACE ]")
    print("  □ Who has write access to the training dataset?")
    print("  □ Is training data sourced from third parties / crawled from the web?")
    print("  □ Are labels generated automatically (crowdsourcing, weak supervision)?")
    print("  □ Is there provenance tracking on training data versions?")
    print("  □ Are there anomaly detection checks on label distributions?")
    print()

    # Model supply chain surface
    print("[ MODEL SUPPLY CHAIN SURFACE ]")
    print("  □ Are pretrained models loaded from external registries (HuggingFace, PyPI)?")
    print("  □ Are model artifacts hash-verified before loading?")
    print("  □ Are pickle-based formats (torch.load, joblib) used without scanning?")
    print("  □ Are model checkpoints stored with access controls?")
    print("  □ Is there a process for scanning models before production deployment?")
    print()

    # Inference surface
    print("[ INFERENCE / DEPLOYMENT SURFACE ]")
    print("  □ Does the API expose confidence scores? (enables membership inference)")
    print("  □ Is there rate limiting to prevent model stealing?")
    print("  □ Is user input sanitized before being passed to model?")
    print("  □ Are inference inputs logged for anomaly detection?")
    print("  □ Can adversarial inputs cause catastrophic compute cost?")
    print()

    # Privacy surface
    print("[ PRIVACY / INFORMATION DISCLOSURE ]")
    print("  □ Does the training set contain PII or regulated data?")
    print("  □ Has the model been evaluated for memorization of training examples?")
    print("  □ Is there a right-to-erasure / unlearning plan for GDPR compliance?")
    print("  □ Can outputs be used to infer sensitive attributes about users?")
    print()

    # Per-component
    print("[ PER-COMPONENT STRIDE QUESTIONS ]")
    for component in components:
        print(f"  {component}:")
        for threat_type in ["Tampered", "DoS target", "Info disclosure", "Privilege escalation"]:
            print(f"    □ Can {component} be {threat_type}?")


# Run the checklist for our RAG example
run_threat_model_checklist(
    system_name="RAG Pipeline (Customer Support)",
    components=["Document ingestion pipeline", "Embedding model", "Vector database", "LLM generator", "API gateway"]
)

## 6. Series Roadmap: Which Notebook Covers What

This series is structured to cover each attack surface in depth, in roughly increasing order of sophistication. Threat modeling (this notebook) is the foundation — every subsequent notebook assumes you can locate the attacks it covers within the broader taxonomy.

| Attack surface | Notebooks |
|----------------|-----------|
| **Inference attacks** | 02 (FGSM), 03 (PGD/C&W), 04 (Defenses), 05 (Membership Inference), 06 (Model Inversion), 07 (Attribute Inference), 12 (Model Stealing) |
| **Training-time attacks** | 08 (Backdoors), 09 (Backdoor Detection), 10 (Data Poisoning) |
| **Supply chain** | 11 (Supply Chain Attacks) |
| **Privacy defenses** | 13 (Differential Privacy Fundamentals), 14 (DP-SGD) |
| **Federated learning** | 15 (FL Basics), 16 (FL Attacks) |
| **Ownership & auditing** | 17 (Model Watermarking) |
| **Tool prototypes** | 18 (`poison-control`), 19 (`model-scan`), 20 (Capstone audit pipeline) |

Each notebook follows the same structure:
1. What is this attack / defense?
2. Why does it work? (intuition before math)
3. Historical context — who discovered it and when
4. From-scratch implementation
5. Practical implications and mitigations
6. Forward pointers to defenses / subsequent notebooks

In [ ]:
# Series overview: attack families mapped to notebooks
print("Series 3 — ML Security: Attack Taxonomy → Notebook Map")
print("=" * 65)
print(f"{'Notebook':>10} {'Attack Family':30} {'Surface':25} {'Goal'}")
print("-" * 95)

for attack in sorted(ATTACK_TAXONOMY, key=lambda a: a.notebook):
    print(
        f"  NB {attack.notebook:02d}  "
        f"{attack.name:30} "
        f"{attack.surface.value:25} "
        f"{attack.attacker_goal[:45]}"
    )

## Summary

ML systems have four distinct attack surfaces, each exploited by different attackers with different capabilities:

```
Surface                  Attacker needs                Primary threats
─────────────────────────────────────────────────────────────────────
Training data            Write access to data          Poisoning, backdoors
Training process         Code/env access               Supply chain, gradient attacks
Model / supply chain     Artifact read/write           Pickle exploits, model tampering
Inference / deployment   Query access                  Adversarial, membership inference
```

**Key principles:**
- Attacks target different pipeline stages — identify which stage before reasoning about defenses
- Characterize attackers by goal (integrity/confidentiality/availability), capability (what they can do), and knowledge (white/gray/black-box)
- STRIDE maps cleanly to ML systems and is useful for communicating with non-ML security teams
- A threat model is a living document — revisit it when the system architecture changes

**Next:** Notebook 02 covers the foundational inference-time attack — adversarial examples via FGSM. This is where the ML security field started, and understanding it deeply is prerequisite to everything that follows.